In [ ]:
"""
Combined spatial benchmark figure.
- Majority-vote color matching: predicted clusters are remapped to ground-truth
  domain colors so that panel colors are semantically consistent.
- If leiden returns the wrong number of clusters, resolution is auto-tuned.
"""

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib import rcParams
import omicverse as ov
import scanpy as sc
import pandas as pd
import numpy as np
import os

rcParams['figure.figsize'] = (4, 4)

# ─── CONFIG ───────────────────────────────────────────────────────────────────
DATA_DIR      = 'Processed_Simulated_Data/Simulated_Dataset_5/'
FIGURE_DIR    = 'Figure/Main/'
FIGURE_NAME   = 'combined_spatial_dataset5'   # output filename (no extension)
DATASET_TITLE = 'Results on simulated spatial multi-omics dataset 5'
os.makedirs(FIGURE_DIR, exist_ok=True)

SPOT_SIZE     = 0.125
N_DOMAINS     = 4                          # expected number of spatial domains
DOMAIN_LABELS = [f'Spatial domain {i+1}' for i in range(N_DOMAINS)]
DOMAIN_COLORS = ['#BFD9E5', '#E7BD39', '#D64F38', '#C7A085']  # domain 1-4

# Ground-truth cluster name → domain index (0-based)
CLUSTER2DOMAIN = {
    'Ery': 0,
    'HMP': 1,
    'HSC': 2,
    'MEP': 3,
}

# (obs_key, display_title, obsm_key, initial_leiden_resolution)
METHODS = [
    ('Scanpy',      'Scanpy',      'scanpy_rna',            0.6 ),
    ('MultiVI',     'MultiVI',     'multivi_multiomics',    0.27),
    ('scGLUE',      'scGLUE',      'scglue_multiomics',     0.6 ),
    ('MUSE',        'MUSE',        'MUSE_multiomics',       0.6 ),
    ('MISO',        'MISO',        'miso_multiomics',       0.6 ),
    ('GraphST',     'GraphST',     'graphst_rna',           0.15),
    ('STAGATE',     'STAGATE',     'stagate_rna',           0.3 ),
    ('Descart',     'Descart',     'Descart_atac',          0.15),
    ('COSMOS',      'COSMOS',      'cosmos_multiomics',     0.1 ),
    ('SpatialGlue', 'SpatialGlue', 'spatialglue_multiomics',0.4 ),
    ('STARNet',     'STARNet',     'stmultigrn_multiomics', 0.6 ),
]

# h5ad file → obsm key inside that file
EMBED_FILES = [
    ('Descart_atac.h5ad',           'x_pca'),
    ('scanpy_rna.h5ad',             'X_pca'),
    ('stagate_rna.h5ad',            'STAGATE'),
    ('graphst_rna.h5ad',            'GraphST_embedding'),
    ('scglue_multiomics.h5ad',      'X_glue'),
    ('multivi_multiomics.h5ad',     'X_multivi'),
    ('spatialglue_multiomics.h5ad', 'SpatialGlue'),
    ('stmultigrn_multiomics.h5ad',  'X_STmultiGAT'),
    ('cosmos_multiomics.h5ad',      'cosmos'),
    ('MUSE_multiomics.h5ad',        'MUSE'),
    ('miso_multiomics.h5ad',        'MISO'),
]
# ─────────────────────────────────────────────────────────────────────────────


def load_embeddings(adata_rna):
    for fname, embed_key in EMBED_FILES:
        fpath = os.path.join(DATA_DIR, fname)
        if not os.path.exists(fpath):
            print(f'[skip] {fname} not found')
            continue
        adata = sc.read_h5ad(fpath)
        dest  = fname.split('.')[0]
        if embed_key not in adata.obsm:
            print(f'[skip] {embed_key} not in {fname}.obsm')
            continue
        if fname == 'stmultigrn_multiomics.h5ad':
            adata_rna.obsm[dest] = adata[adata_rna.obs_names, :].obsm[embed_key]
        else:
            adata_rna.obsm[dest] = adata.obsm[embed_key]
    return adata_rna


def cluster_to_n(adata, obsm_key, target_n, init_res, n_neighbors=15):
    """
    Run leiden clustering and binary-search resolution until we get exactly
    `target_n` clusters (or the closest possible). Returns the cluster Series.
    """
    n_pcs = adata.obsm[obsm_key].shape[1]
    ov.pp.neighbors(adata, n_neighbors=n_neighbors, n_pcs=n_pcs, use_rep=obsm_key)

    res            = init_res
    lo, hi         = 0.0, 5.0
    best_labels    = None
    best_diff      = 999

    for _ in range(30):
        ov.utils.cluster(adata, method='leiden', resolution=res)
        labels = adata.obs['leiden'].copy()
        n      = labels.nunique()
        diff   = abs(n - target_n)

        if diff < best_diff:
            best_diff, best_labels = diff, labels.copy()

        if n == target_n:
            break
        elif n < target_n:
            lo  = res
            res = (res + hi) / 2
        else:
            hi  = res
            res = (res + lo) / 2

        if hi - lo < 1e-4:
            break

    if best_diff > 0:
        print(f'  [warn] {obsm_key}: settled on {best_labels.nunique()} clusters '
              f'(target {target_n})')
    return best_labels


def majority_vote_remap(pred_labels, gt_domain_idx):
    """
    Greedy majority-vote remapping of predicted cluster IDs to ground-truth
    domain indices. Each predicted cluster is matched to the gt domain with
    the greatest overlap.

    Critically, all N_DOMAINS category levels are always set on the returned
    Series (even if some are empty) so that scanpy assigns colors by position
    consistently across every panel.

    Returns
    -------
    remapped_labels : pd.Series  – category labels 'domain_0' … 'domain_N'
    ordered_colors  : list[str]  – DOMAIN_COLORS in scanpy's sorted-cat order
    """
    pred_unique = sorted(pred_labels.unique(), key=lambda x: int(x))
    n_pred = len(pred_unique)
    n_gt   = N_DOMAINS

    # Contingency matrix [n_pred × n_gt]
    cont = np.zeros((n_pred, n_gt), dtype=int)
    for p_idx, p_label in enumerate(pred_unique):
        mask = pred_labels == p_label
        for d in range(n_gt):
            cont[p_idx, d] = (gt_domain_idx[mask] == d).sum()

    # Greedy assignment: strongest signal first
    pred2gt = {}
    used_gt = set()
    order   = sorted(range(n_pred), key=lambda i: cont[i].max(), reverse=True)
    for p_idx in order:
        available = [d for d in range(n_gt) if d not in used_gt]
        if not available:
            available = list(range(n_gt))
        best_d = max(available, key=lambda d: cont[p_idx, d])
        pred2gt[pred_unique[p_idx]] = best_d
        used_gt.add(best_d)

    # Remap and FORCE all N_DOMAINS levels to be present — this ensures
    # scanpy's color list is always indexed as domain_0→color0, domain_1→color1 …
    all_cats = [f'domain_{d}' for d in range(n_gt)]
    remapped = (
        pred_labels
        .map(lambda x: f'domain_{pred2gt[x]}')
        .astype('category')
        .cat.set_categories(all_cats)
    )

    # Colors are now always in fixed order domain_0…domain_{N-1}
    ordered_colors = list(DOMAIN_COLORS[:n_gt])

    return remapped, ordered_colors


def run_all_methods(adata_rna):
    gt_domain_idx = adata_rna.obs['ground_truth'].map(CLUSTER2DOMAIN)

    for obs_key, title, obsm_key, init_res in METHODS:
        if obsm_key not in adata_rna.obsm:
            print(f'[skip] {obsm_key} not in obsm')
            continue
        print(f'Clustering {title} …')
        raw              = cluster_to_n(adata_rna, obsm_key, N_DOMAINS, init_res)
        remapped, colors = majority_vote_remap(raw, gt_domain_idx)
        adata_rna.obs[obs_key]             = remapped
        adata_rna.uns[f'{obs_key}_colors'] = colors

    return adata_rna


def _trim_ax(ax):
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    ax.set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))
    ax.set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
    ax.set_xlabel('')
    ax.set_ylabel('')


def plot_panel(adata, color_key, title, ax):
    tmp = sc.pl.spatial(
        adata,
        color=[color_key],
        colorbar_loc=None,
        legend_loc=None,
        spot_size=SPOT_SIZE,
        legend_fontsize=15,
        ax=ax,
        show=False,
    )
    panel_ax = tmp[0] if isinstance(tmp, list) else ax
    panel_ax.set_title(title, fontsize=18, fontweight='bold', pad=8)
    _trim_ax(panel_ax)
    return panel_ax


def build_figure(adata_rna):
    n_cols       = 6
    panel_size   = 2.8
    legend_width = 2.2

    # Derive row count from actual panel count
    all_panels = (
        [('ground_truth_plot', 'Ground truth')]
        + [(obs_key, title) for obs_key, title, *_ in METHODS]
    )
    import math
    n_rows = math.ceil(len(all_panels) / n_cols)

    fig_w = n_cols * panel_size + legend_width
    fig_h = n_rows * panel_size + 1.0

    fig = plt.figure(figsize=(fig_w, fig_h))

    gs = gridspec.GridSpec(
        n_rows, n_cols + 1,
        width_ratios=[1]*n_cols + [legend_width / panel_size],
        hspace=0.30,
        wspace=0.06,
        top=0.88, bottom=0.02, left=0.008, right=0.995,
    )

    for idx, (obs_key, title) in enumerate(all_panels):
        row = idx // n_cols
        col = idx %  n_cols
        ax  = fig.add_subplot(gs[row, col])
        if obs_key not in adata_rna.obs.columns:
            ax.axis('off')
            ax.set_title(title, fontsize=18, fontweight='bold')
            continue
        plot_panel(adata_rna, obs_key, title, ax)

    # ── Legend ──
    leg_ax = fig.add_subplot(gs[:, n_cols])
    leg_ax.axis('off')
    patches = [
        mpatches.Patch(facecolor=DOMAIN_COLORS[i], edgecolor='none',
                       label=DOMAIN_LABELS[i])
        for i in range(N_DOMAINS)
    ]
    leg_ax.legend(
        handles=patches,
        loc='center left',
        fontsize=23,
        frameon=False,
        handlelength=1.4,
        handleheight=1.4,
        borderpad=0,
        labelspacing=0.9,
    )

    # ── Title + rule ──
    fig.suptitle(DATASET_TITLE, fontsize=20, fontweight='bold', y=0.995)
    line = plt.Line2D(
        [0.005, 0.995], [0.950, 0.950],
        transform=fig.transFigure,
        color='black', linewidth=1.0,
    )
    fig.add_artist(line)

    for ext, dpi in [('pdf', 300), ('png', 150)]:
        out = os.path.join(FIGURE_DIR, f'{FIGURE_NAME}.{ext}')
        fig.savefig(out, dpi=dpi, bbox_inches='tight')
        print(f'Saved → {out}')
    plt.close(fig)


# ─── MAIN ────────────────────────────────────────────────────────────────────
adata_rna = sc.read_h5ad(os.path.join(DATA_DIR, 'scanpy_rna.h5ad'))

# Ground truth: use domain_X naming for color-index consistency
gt_domain_idx = adata_rna.obs['ground_truth'].map(CLUSTER2DOMAIN)
all_cats = [f'domain_{d}' for d in range(N_DOMAINS)]
gt_mapped = gt_domain_idx.map(lambda i: f'domain_{int(i)}' if pd.notna(i) else np.nan)
adata_rna.obs['ground_truth_plot'] = (
    gt_mapped.astype('category')
             .cat.set_categories(all_cats)
)
adata_rna.uns['ground_truth_plot_colors'] = list(DOMAIN_COLORS)

adata_rna = load_embeddings(adata_rna)
adata_rna = run_all_methods(adata_rna)
build_figure(adata_rna)

adata_rna.write_h5ad(os.path.join(DATA_DIR, 'adata_all.h5ad'))
print('Done.')